# 5. Explainable AI: Grad-CAM Radiologic Saliency Visualization

**Axlero Diagnostic Systems**  
**Project:** Healthcare & Medical – Diagnostic Lab Image Classification  
**Phase 4:** Gradient-Weighted Class Activation Mapping (Grad-CAM) on DenseNet-121.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.gradcam import GradCAM
from src.model import build_model, get_gradcam_target_layer
from src.preprocessing import DEFAULT_IMAGE_SIZE, load_image, preprocess_pil_image
from src.utils import get_device

device = get_device()
print(f"Active Device: {device}")

## 1. Load Production DenseNet-121 Architecture & Target Conv Layer
Target layer: `backbone.features.denseblock4.denselayer16`.

In [ ]:
model = build_model(architecture='densenet121', num_classes=2, pretrained=True, freeze_backbone=True)
model = model.to(device)
target_layer = get_gradcam_target_layer(model, 'densenet121')
gradcam = GradCAM(model, target_layer)
print("Grad-CAM successfully initialized with target layer hooks.")

## 2. Explain Pneumonia Radiograph Prediction
Loads a representative pneumonia test scan and visualizes feature activation maps.

In [ ]:
test_pneu = project_root / 'data' / 'raw' / 'chest_xray' / 'test' / 'PNEUMONIA' / 'person100_bacteria_475.jpeg'
out_fig = project_root / 'reports' / 'figures' / 'gradcam_sample_pneumonia.png'
res = gradcam.visualize_side_by_side(test_pneu, out_fig, true_label='PNEUMONIA')
print(f"Prediction: {res['predicted_class']} (Confidence: {res['confidence']*100:.1f}%)")
print(f"Risk Tier: {res['risk_tier']}")

## 3. Explain Normal Lung Radiograph Prediction

In [ ]:
test_norm = project_root / 'data' / 'raw' / 'chest_xray' / 'test' / 'NORMAL' / 'IM-0001-0001.jpeg'
out_norm = project_root / 'reports' / 'figures' / 'gradcam_sample_normal.png'
res_n = gradcam.visualize_side_by_side(test_norm, out_norm, true_label='NORMAL')
print(f"Prediction: {res_n['predicted_class']} (Confidence: {res_n['confidence']*100:.1f}%)")